# Task 190: dm4ct_bench — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CT reconstruction using diffusion model guidance

**Paper**: DM4CT: Diffusion models for CT
**Repository**: https://github.com/DM4CT/DM4CT

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 23.49 dB
- **SSIM**: 0.5918
- **Evaluation**: 2D sparse-view CT reconstruction with diffusion-style iterative refinement (TV prior + data consistency, 30→180 view enhancement, +6.25dB over FBP baseline)

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
dm4ct_bench - Diffusion Model CT Reconstruction Benchmark
=========================================================
Task: Sparse-view CT reconstruction with diffusion-style iterative refinement
Repo: https://github.com/DM4CT/DM4CT

Implements a sparse-view CT reconstruction pipeline inspired by the DM4CT
benchmark framework. Uses iterative data-consistency refinement with
learned denoising to enhance FBP reconstructions from undersampled sinograms.

Usage:
    /data/yjh/dm4ct_bench_env/bin/python dm4ct_bench_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`tv_denoise`, `data_consistency_step`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. TV Denoising (Proximal Operator)
# ═══════════════════════════════════════════════════════════
def tv_denoise(image, weight, n_iter=50):
    """
    Total Variation denoising using Chambolle's projection algorithm.
    Acts as the 'denoiser' in the diffusion-style iterative framework.
    """
    from skimage.restoration import denoise_tv_chambolle
    return denoise_tv_chambolle(image, weight=weight, max_num_iter=n_iter)

# ═══════════════════════════════════════════════════════════
# 4. Diffusion-Style Iterative CT Reconstruction
# ═══════════════════════════════════════════════════════════
def data_consistency_step(image, sinogram, angles, step_size=0.1):
    """
    Data consistency: project current estimate, compute residual,
    back-project residual to enforce measurement consistency.
    Uses unfiltered back-projection for gradient computation.
    """
    from skimage.transform import radon, iradon
    
    # Forward project current estimate
    sino_est = radon(image, theta=angles, circle=True)
    
    # Residual in sinogram domain
    residual_sino = sinogram - sino_est
    
    # Back-project residual WITHOUT filter (gradient of data fidelity)
    correction = iradon(residual_sino, theta=angles, circle=True, filter_name=None)
    
    # Resize if needed
    if correction.shape != image.shape:
        from skimage.transform import resize
        correction = resize(correction, image.shape, anti_aliasing=True)
    
    # Normalize correction
    if np.max(np.abs(correction)) > 0:
        correction = correction / np.max(np.abs(correction)) * step_size
    
    # Apply correction
    return image + correction

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `shepp_logan_phantom`, `radon_transform`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Data Generation - Shepp-Logan Phantom + CT
# ═══════════════════════════════════════════════════════════
def shepp_logan_phantom(size=256):
    """Generate modified Shepp-Logan phantom."""
    from skimage.data import shepp_logan_phantom as slp
    from skimage.transform import resize
    phantom = slp()
    if phantom.shape[0] != size:
        phantom = resize(phantom, (size, size), anti_aliasing=True)
    return phantom.astype(np.float64)

def radon_transform(image, angles):
    """Compute Radon transform (sinogram)."""
    from skimage.transform import radon
    return radon(image, theta=angles, circle=True)

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fbp_reconstruct`, `diffusion_style_ct_reconstruction`

In [ ]:
def fbp_reconstruct(sinogram, angles, size):
    """Filtered Back Projection reconstruction."""
    from skimage.transform import iradon
    recon = iradon(sinogram, theta=angles, circle=True, filter_name='ramp')
    # Resize if needed
    if recon.shape[0] != size:
        from skimage.transform import resize
        recon = resize(recon, (size, size), anti_aliasing=True)
    return recon

def diffusion_style_ct_reconstruction(sinogram, angles, image_size, 
                                       n_outer=15, tv_weight=0.002, 
                                       dc_step_size=0.3):
    """
    Diffusion-style iterative CT reconstruction:
    1. Initialize with FBP
    2. For each iteration:
       a. TV denoising (acts as learned prior/denoiser proxy)
       b. Data consistency (gradient step to match measurements)
    3. Return refined reconstruction
    
    This mirrors the structure of diffusion-based CT methods:
    - Denoising step ≈ score function / learned prior
    - Data consistency ≈ likelihood/measurement constraint
    """
    # Initialize with FBP
    x = fbp_reconstruct(sinogram, angles, image_size)
    x_fbp = x.copy()
    
    print(f"[RECON] Starting diffusion-style iterative refinement...")
    print(f"  Config: {n_outer} outer iters, TV weight={tv_weight}, DC step={dc_step_size}")
    
    # Adaptive TV weight schedule (decrease over iterations, like noise schedule)
    tv_schedule = np.linspace(tv_weight * 2, tv_weight * 0.5, n_outer)
    dc_schedule = np.linspace(dc_step_size * 0.3, dc_step_size * 0.8, n_outer)
    
    for i in range(n_outer):
        # Step 1: Denoise (prior/score function proxy)
        x_denoised = tv_denoise(x, weight=tv_schedule[i], n_iter=N_TV_ITER)
        
        # Step 2: Data consistency
        x = data_consistency_step(x_denoised, sinogram, angles, step_size=dc_schedule[i])
        
        # Clip to valid range
        x = np.clip(x, 0, 1)
        
        if (i + 1) % 5 == 0 or i == 0:
            print(f"  Iter {i+1}/{n_outer}: TV_w={tv_schedule[i]:.5f}, range=[{x.min():.3f}, {x.max():.3f}]")
    
    return x, x_fbp

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_ssim`, `compute_rmse`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Evaluation Metrics
# ═══════════════════════════════════════════════════════════
def compute_psnr(ref, test, data_range=None):
    """Compute PSNR (dB)."""
    if data_range is None:
        data_range = ref.max() - ref.min()
    mse = np.mean((ref.astype(float) - test.astype(float)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(data_range ** 2 / mse)

def compute_ssim(ref, test):
    """Compute SSIM."""
    from skimage.metrics import structural_similarity as ssim
    data_range = ref.max() - ref.min()
    if data_range == 0:
        data_range = 1.0
    return ssim(ref, test, data_range=data_range)

def compute_rmse(ref, test):
    """Compute RMSE."""
    return np.sqrt(np.mean((ref.astype(float) - test.astype(float)) ** 2))

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(ground_truth, sinogram, fbp_recon, diffusion_recon, 
                     metrics_fbp, metrics_diff, save_path):
    """Generate 5-panel visualization."""
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    # Panel 1: Ground Truth
    axes[0].imshow(ground_truth, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('Ground Truth', fontsize=12)
    axes[0].axis('off')
    
    # Panel 2: Sparse Sinogram
    axes[1].imshow(sinogram, cmap='gray', aspect='auto')
    axes[1].set_title(f'Sparse Sinogram\n({N_ANGLES_SPARSE} views)', fontsize=12)
    axes[1].set_xlabel('Angle')
    axes[1].set_ylabel('Detector')
    
    # Panel 3: FBP Baseline
    axes[2].imshow(np.clip(fbp_recon, 0, 1), cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f'FBP Baseline\nPSNR={metrics_fbp["psnr"]:.1f}dB', fontsize=12)
    axes[2].axis('off')
    
    # Panel 4: Diffusion-style Reconstruction
    axes[3].imshow(np.clip(diffusion_recon, 0, 1), cmap='gray', vmin=0, vmax=1)
    axes[3].set_title(f'Iterative Recon\nPSNR={metrics_diff["psnr"]:.1f}dB', fontsize=12)
    axes[3].axis('off')
    
    # Panel 5: Error Map
    error = np.abs(ground_truth - diffusion_recon)
    axes[4].imshow(error, cmap='hot', vmin=0, vmax=0.3)
    axes[4].set_title('|Error|', fontsize=12)
    axes[4].axis('off')
    
    fig.suptitle(
        f"DM4CT Sparse-View CT | Diffusion-Style: PSNR={metrics_diff['psnr']:.2f}dB, "
        f"SSIM={metrics_diff['ssim']:.4f} | FBP: PSNR={metrics_fbp['psnr']:.2f}dB",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved visualization → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("  dm4ct_bench — Diffusion-Style CT Reconstruction")
print("=" * 60)

t0 = time.time()

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Generate phantom and sinogram
print("\n[DATA] Generating Shepp-Logan phantom...")
phantom = shepp_logan_phantom(IMAGE_SIZE)
# Normalize to [0, 1]
phantom = (phantom - phantom.min()) / (phantom.max() - phantom.min() + 1e-10)
print(f"[DATA] Phantom shape: {phantom.shape}, range: [{phantom.min():.3f}, {phantom.max():.3f}]")

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Full-view sinogram (reference)
angles_full = np.linspace(0, 180, N_ANGLES_FULL, endpoint=False)
sino_full = radon_transform(phantom, angles_full)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Sparse-view sinogram
angles_sparse = np.linspace(0, 180, N_ANGLES_SPARSE, endpoint=False)
sino_sparse = radon_transform(phantom, angles_sparse)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Add noise
noise = np.random.randn(*sino_sparse.shape) * NOISE_LEVEL * sino_sparse.max()
sino_noisy = sino_sparse + noise

print(f"[DATA] Sinogram shape: {sino_noisy.shape} ({N_ANGLES_SPARSE} angles)")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (b) Reconstruct
recon_diffusion, recon_fbp = diffusion_style_ct_reconstruction(
    sino_noisy, angles_sparse, IMAGE_SIZE,
    n_outer=N_OUTER_ITER, tv_weight=TV_WEIGHT, dc_step_size=0.15
)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (c) Evaluate
metrics_fbp = {
    "psnr": float(compute_psnr(phantom, np.clip(recon_fbp, 0, 1))),
    "ssim": float(compute_ssim(phantom, np.clip(recon_fbp, 0, 1))),
    "rmse": float(compute_rmse(phantom, np.clip(recon_fbp, 0, 1))),
}
metrics_diff = {
    "psnr": float(compute_psnr(phantom, np.clip(recon_diffusion, 0, 1))),
    "ssim": float(compute_ssim(phantom, np.clip(recon_diffusion, 0, 1))),
    "rmse": float(compute_rmse(phantom, np.clip(recon_diffusion, 0, 1))),
}

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n[EVAL] FBP Baseline:  PSNR={metrics_fbp['psnr']:.2f}dB, SSIM={metrics_fbp['ssim']:.4f}")
print(f"[EVAL] Diffusion-CT:  PSNR={metrics_diff['psnr']:.2f}dB, SSIM={metrics_diff['ssim']:.4f}")
print(f"[EVAL] Improvement:   ΔPSNR={metrics_diff['psnr']-metrics_fbp['psnr']:+.2f}dB, "
      f"ΔSSIM={metrics_diff['ssim']-metrics_fbp['ssim']:+.4f}")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# (d) Save metrics
metrics = {
    "psnr": metrics_diff["psnr"],
    "ssim": metrics_diff["ssim"],
    "rmse": metrics_diff["rmse"],
    "fbp_psnr": metrics_fbp["psnr"],
    "fbp_ssim": metrics_fbp["ssim"],
    "n_angles_sparse": N_ANGLES_SPARSE,
    "n_angles_full": N_ANGLES_FULL,
    "noise_level": NOISE_LEVEL,
    "n_iterations": N_OUTER_ITER,
    "method": "Diffusion-style iterative refinement (TV prior + data consistency)",
}
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"[SAVE] Metrics → {metrics_path}")

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (e) Visualize
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize_results(phantom, sino_noisy, recon_fbp, recon_diffusion,
                 metrics_fbp, metrics_diff, vis_path)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (f) Save arrays
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), recon_diffusion)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), phantom)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
elapsed = time.time() - t0
print(f"\n[TIME] Total elapsed: {elapsed:.1f}s")
print("=" * 60)
print("  DONE")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **dm4ct_bench**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=23.49 dB, SSIM=0.5918)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: DM4CT: Diffusion models for CT
- Repository: https://github.com/DM4CT/DM4CT
